<a href="https://colab.research.google.com/github/bananaduck01/Twitter-Bot/blob/main/DS_Datathon_LULC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Packages
%%capture
!pip install rioxarray stackstac pystac-client planetary-computer
!pip install earthengine-api
!pip install rasterio
!pip install pandas
!pip install matplotlib
!pip install seaborn


In [ ]:
# ONLY NEED TO RUN THIS ONCE
from google.colab import drive

# Mounting drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Checking that your mount worked
import os

# If you saved to Google Drive
if os.path.exists('/content/drive/MyDrive/'):
    print("Files in your Drive:")
    print(os.listdir('/content/drive/MyDrive/'))
else:
    print("Drive not mounted. Run: from google.colab import drive; drive.mount('/content/drive')")
    #from google.colab import drive; drive.mount('/content/drive')

Files in your Drive:
['My stuff', 'School', 'image (2).jpg', 'image (1).jpg', 'Extra-ordinary Heroes.gdoc', 'Untitled drawing (1).gdraw', 'Hi.gdoc', 'Pep Talk.gdoc', 'image.jpg', 'Untitled drawing.gdraw', '(ᵔᴥᵔ).gdoc', '¯\\_(ツ)_ ¯.gdoc', '....gdoc', 'ʕ•ᴥ•ʔ.gdoc', 'Exe.gdoc', 'I just wanted to say....gdoc', 'One word phrase based.gdoc', '🌃.gdoc', '(づ｡◕‿‿◕｡)づ.gdoc', 'Untitled document (21).gdoc', 'Character Profiles.gdoc', 'Zairose Universe.gdoc', 'Song inspired plots.gdoc', 'F*ck It.gdoc', 'Untitled document (20).gdoc', 'Oooh ma gawd is this a script I see.gdoc', 'IMG_1637 (1).TRIM.MOV', 'IMG_1637.TRIM.MOV', 'Convos.gdoc', 'That Game Show Life.gdoc', 'Copy of That Game Show Life(Backup).gdoc', 'Fido.gdoc', 'Untitled.gdoc', 'Title base.gdoc', 'Shits n giggles.gdoc', 'The dream.gdoc', 'Temp.gdoc', 'Story baby.gdoc', 'The Good.gdoc', 'Things I wanted to say.gdoc', 'Untitled document (19).gdoc', 'Things.gdoc', 'Reasons to live another day.gdoc', 'Cookie recipe.gdoc', 'B.gdoc', 'Emotions.g

In [ ]:
import ee
import geemap

# Google Cloud
project_number = 456110130984
project_id = "ds-club-lulc"

# 1. Authenticate (Each user must do this once per session)
ee.Authenticate(force=True)

# 2. Initialize with your specific project
ee.Initialize(project=project_id)

# 3. Load the ESA WorldCover 2020 dataset
# This is a global image, so we just take the first (and only) one in the collection
dataset = ee.ImageCollection("ESA/WorldCover/v100").first()

# 4. Create an interactive map
Map = geemap.Map()
Map.add_basemap('SATELLITE')

# Define visualization parameters (standard colors for land cover)
viz_params = {'bands': ['Map']}

Map.addLayer(dataset, viz_params, "ESA WorldCover 2020")
Map.add_colorbar(viz_params, label="Land Cover Class")

Map

EEException: Caller does not have required permission to use project ds-club-lulc. Grant the caller the roles/serviceusage.serviceUsageConsumer role, or a custom role with the serviceusage.services.use permission, by visiting https://console.developers.google.com/iam-admin/iam/project?project=ds-club-lulc and then retry. Propagation of the new permission may take a few minutes.

In [ ]:
from google.colab import auth
import pandas as pd

# Authenticating
auth.authenticate_user()

# Creating training and validating(testing) pathways
train_path = 'gs://dslulc-training-data/land_use_train.csv'
val_path = 'gs://dslulc-training-data/land_use_val.csv'

#inserting training and validation (testing) pathways into dataframes
df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)

# Define features (bands) and label
bands = ['B2', 'B3', 'B4', 'B8']
X_train = df_train[bands]
y_train = df_train['lc']

X_val = df_val[bands]
y_val = df_val['lc']

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Model creation and training
# Increase enseble to 200-500 later prolly (for stability), 100 is good for testing
model = RandomForestClassifier(n_estimators=500, random_state=42)
model.fit(X_train, y_train)
model


In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt

# 1. Load the GeoTIFF image
with rasterio.open('/content/drive/MyDrive/DSTestImages/Test_Image_SanDiego.tif') as src:
    img = src.read()  # Shape: [Bands, Height, Width]
    profile = src.profile  # Stores geospatial metadata for later use

# Confirm the image has the expected number of spectral bands
if img.shape[0] != 4:
    raise ValueError(f"Expected 4 bands (B2, B3, B4, B8), but got {img.shape[0]} bands")

# Convert raster values to float for compatibility with the trained model
img = img.astype(np.float32)

# 2. Prepare the data for prediction
# Rearrange from (Bands, Height, Width) to (Height, Width, Bands),
# then flatten into (Number of pixels, 4 bands)
h, w = img.shape[1], img.shape[2]
img_reshaped = img.transpose(1, 2, 0).reshape(-1, 4)

# Replace any invalid values (NaN or Inf) with zero before prediction
img_reshaped = np.nan_to_num(img_reshaped, nan=0.0, posinf=0.0, neginf=0.0)

# 3. Run the trained model on every pixel
prediction = model.predict(img_reshaped)

# Verify that the number of predictions matches the number of pixels
if prediction.shape[0] != h * w:
    raise ValueError(
        f"Prediction size {prediction.shape[0]} does not match image pixels {h*w}"
    )

# 4. Reshape predictions back into the original image grid
prediction_map = prediction.reshape(h, w)

# 5. Visualize the classified map
plt.imshow(prediction_map, cmap='terrain')
plt.colorbar(label='Land Use Class')
plt.show()





In [ ]:
# 1. Save the prediction locally in Colab first using the original profile
# 1. Trigger the authentication flow
ee.Authenticate()

# 2. Initialize with your specific project ID
# This links your session to the 'ds-club-lulc' project
ee.Initialize(project='ds-club-lulc')


new_profile = profile.copy()
new_profile.update(dtype=rasterio.int32, count=1, compress='lzw')

local_file = 'SD_Prediction.tif'
with rasterio.open(local_file, 'w', **new_profile) as dst:
    dst.write(prediction_map.astype(np.int32), 1)

# 2. Upload the file to your GCS Bucket (using gsutil)
bucket_name = "dslulc-training-data"
!gsutil cp {local_file} gs://{bucket_name}/



# 1. Define your variables
asset_id = 'projects/ds-club-lulc/assets/Bakersfield_Test_Prediction'
gs_uri = f'gs://{bucket_name}/{local_file}'

# 2. Run the upload command
# This tells GEE: "Grab this file from my bucket and make it an asset"
#!earthengine upload image --asset_id={asset_id} {gs_uri}

#print(f"Task started! Go to https://code.earthengine.google.com/ and check the 'Tasks' tab.")

In [ ]:
import joblib
import rasterio
import numpy as np

# 1. LOAD THE BRAIN (Done only once)
# This loads the weights and trees exactly as they were after training
model = joblib.load('/content/drive/MyDrive/DSTestImages/landcover_rf_model.pkl')

def classify_new_city(file_path):
    # 2. LOAD THE NEW IMAGE
    with rasterio.open(file_path) as src:
        img = src.read().astype(np.float32)
        profile = src.profile

    # 3. MATCH THE FEATURES (Essential!)
    # If you trained on B2, B3, B4, B8, and NDVI, you MUST calculate NDVI here
    nir, red = img[3], img[2]
    ndvi = (nir - red) / (nir + red + 1e-10)

    # Stack to match the exact 5-column format the model expects
    input_data = np.concatenate([img, ndvi[np.newaxis, :]], axis=0)

    # Reshape for the model
    h, w = img.shape[1], img.shape[2]
    pixels = input_data.transpose(1, 2, 0).reshape(-1, 5)
    pixels = np.nan_to_num(pixels)

    # 4. PREDICT (Using existing knowledge)
    prediction = model.predict(pixels)
    return prediction.reshape(h, w)

In [ ]:
# Testing

from xgboost import XGBClassifier
XGB_model = XGBClassifier( n_estimators=200, max_depth=6, learning_rate=0.1 )
XGB_model.fit(X_train, y_train)
XGB_model